# WaveForge — Brain Haemorrhage Dataset Generator

**Just click Run All — fully automated. Both GPUs run in parallel.**

Generates a medically realistic FDTD simulation dataset with:
- Every sample uses a **unique randomised head geometry** (population distribution)
- Frequency-dependent tissue properties via Cole-Cole model (Gabriel 1996)
- Anatomically constrained bleed placement
- Blood aging: acute / subacute / chronic
- **GPU[0] and GPU[1] run simultaneously** (Python threading)

| Property | Value |
|----------|-------|
| Frequency | 1.0 GHz | Grid | 64³ at 3mm/cell (192mm domain) |
| Antennas | 8-element ring | Steps | 300 per TX |
| Classes | 0=healthy 1=epidural 2=subdural 3=intracerebral |
| Train set | 1600 samples, unique phantoms, GPU[0] |
| Test set | 400 samples, **separate seed space**, GPU[1] |

**Accelerator:** GPU T4 x2 | **Estimated time:** ~10h total (parallel)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, pathlib, threading, time, json, datetime

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/kaggle/working/waveforge')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

import torch, numpy as np
assert torch.cuda.is_available(), 'No GPU — enable T4 x2 accelerator!'
N_GPUS = torch.cuda.device_count()
print(f'GPUs available: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name} {p.total_memory/1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print('✅ Setup complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Configuration                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
FREQ_GHZ    = 1.0
GRID_SIZE   = 64
DX_MM       = 3.0
N_TX        = 8
RING_RADIUS = 30
N_STEPS     = 300

N_TRAIN     = 1600   # train samples  (unique phantoms, GPU[0])
N_TEST      = 400    # test samples   (separate seed space, GPU[1])

OUTPUT_ROOT = pathlib.Path('/kaggle/working/brain_haemorrhage_dataset')
TRAIN_DIR   = OUTPUT_ROOT / 'train'
TEST_DIR    = OUTPUT_ROOT / 'test'

TRAIN_SEED  = 0
TEST_SEED   = 10_000_000   # separate seed space → no phantom overlap with train

GPU0 = 'cuda:0'
GPU1 = 'cuda:1' if N_GPUS > 1 else 'cuda:0'

# Time estimate
sec_per_sample = N_TX * N_STEPS * GRID_SIZE**3 / 130e6 * 2  # conservative T4 estimate
parallel_h = sec_per_sample * max(N_TRAIN, N_TEST) / 3600
print(f'Config: {GRID_SIZE}³ grid, {FREQ_GHZ}GHz, {N_TX} antennas, {N_STEPS} steps')
print(f'Train: {N_TRAIN} samples on {GPU0}')
print(f'Test:  {N_TEST} samples on {GPU1}')
print(f'Est:   ~{sec_per_sample:.0f}s/sample → ~{parallel_h:.1f}h wall time (parallel on 2 GPUs)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Validate 4 samples (one per class) before full run           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from datasets.generator import BrainDatasetGenerator

print('Quick validation — 4 samples (one per class)...')
val_gen = BrainDatasetGenerator(
    output_dir=str(OUTPUT_ROOT / 'validation'),
    freq_hz=FREQ_GHZ * 1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
    n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
    device=GPU0,
)
val_manifest = val_gen.generate_balanced_dataset(
    n_samples=4, phantom_id='train', base_seed=42, show_progress=True
)

if val_manifest['n_completed'] < 2:
    raise RuntimeError('Validation failed — fewer than 2 samples generated. Check GPU.')

s = np.load(val_manifest['sample_paths'][0], allow_pickle=True)
print(f'\nSample check:')
print(f'  signals_scattered: {s["signals_scattered"].shape}')
print(f'  das_image: {s["das_image"].shape}')
print(f'  label={s["label"]} type={s["bleed_type"]} age={s["bleed_age"]}')
print(f'  signal_energy: {(s["signals_scattered"]**2).sum():.3e}')
print('\n✅ Validation passed — launching parallel generation on both GPUs')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Parallel generation: GPU[0]=train, GPU[1]=test               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Both generators run simultaneously in separate Python threads.
# Each thread owns one CUDA device — no locking needed.

results = {}   # shared dict — threads write their manifests here
errors  = {}   # any exceptions land here

def run_train():
    try:
        print(f'[GPU0] Starting {N_TRAIN} train samples...')
        gen = BrainDatasetGenerator(
            output_dir=str(TRAIN_DIR),
            freq_hz=FREQ_GHZ * 1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
            n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
            device=GPU0, seed=TRAIN_SEED,
        )
        results['train'] = gen.generate_balanced_dataset(
            n_samples=N_TRAIN, phantom_id='train',
            base_seed=TRAIN_SEED, show_progress=True,
        )
        print(f'[GPU0] Train done: {results["train"]["n_completed"]} samples')
    except Exception as e:
        errors['train'] = e
        print(f'[GPU0] ERROR: {e}')

def run_test():
    try:
        print(f'[GPU1] Starting {N_TEST} test samples (independent seed space)...')
        gen = BrainDatasetGenerator(
            output_dir=str(TEST_DIR),
            freq_hz=FREQ_GHZ * 1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
            n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
            device=GPU1, seed=TEST_SEED,
        )
        results['test'] = gen.generate_balanced_dataset(
            n_samples=N_TEST, phantom_id='test',
            base_seed=TEST_SEED, show_progress=True,
        )
        print(f'[GPU1] Test done: {results["test"]["n_completed"]} samples')
    except Exception as e:
        errors['test'] = e
        print(f'[GPU1] ERROR: {e}')

t_wall = time.time()

t_train = threading.Thread(target=run_train, daemon=True)
t_test  = threading.Thread(target=run_test,  daemon=True)

t_train.start()
t_test.start()

# Wait for both to finish
t_train.join()
t_test.join()

wall_h = (time.time() - t_wall) / 3600

if errors:
    for k, e in errors.items():
        print(f'ERROR in {k}: {e}')
    raise RuntimeError('Generation failed — see errors above')

train_manifest = results['train']
test_manifest  = results['test']

print(f'\n✅ Both GPUs finished in {wall_h:.2f}h')
print(f'  Train: {train_manifest["n_completed"]} samples, classes: {train_manifest["class_counts"]}')
print(f'  Test:  {test_manifest["n_completed"]} samples, classes: {test_manifest["class_counts"]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Save master manifest                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
try:
    commit = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True
    ).strip()
except Exception:
    commit = 'unknown'

master = {
    'version': '1.1',
    'created_at': datetime.datetime.now().isoformat(),
    'waveforge_commit': commit,
    'n_total':  train_manifest['n_completed'] + test_manifest['n_completed'],
    'n_train':  train_manifest['n_completed'],
    'n_test':   test_manifest['n_completed'],
    'train_class_counts': train_manifest['class_counts'],
    'test_class_counts':  test_manifest['class_counts'],
    'class_names': {0: 'healthy', 1: 'epidural', 2: 'subdural', 3: 'intracerebral'},
    'phantom_design': 'unique_per_sample',  # NOT fixed A/B — each sample has own geometry
    'train_seed_space': f'seeds 0 to {N_TRAIN * 10}',
    'test_seed_space':  f'seeds 10_000_000 to {10_000_000 + N_TEST * 10}',
    'freq_hz': FREQ_GHZ * 1e9,
    'grid_shape': [GRID_SIZE, GRID_SIZE, GRID_SIZE],
    'dx_mm': DX_MM, 'n_tx': N_TX, 'n_rx': N_TX, 'n_steps': N_STEPS,
    'train_dir': str(TRAIN_DIR),
    'test_dir':  str(TEST_DIR),
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'dataset_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(master, f, indent=2)

print(f'Manifest saved: {manifest_path}')
print(f'Total samples: {master["n_total"]}  |  commit: {commit}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Visualise: signals + DAS for one sample per class            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

label_to_name = {0:'Healthy', 1:'Epidural', 2:'Subdural', 3:'Intracerebral'}
samples_by_label = {}
for path, label in zip(train_manifest['sample_paths'], train_manifest['labels']):
    if label not in samples_by_label:
        samples_by_label[label] = np.load(path, allow_pickle=True)
    if len(samples_by_label) == 4: break

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
fig.suptitle('WaveForge Brain Haemorrhage Dataset — One Sample per Class', fontsize=13, fontweight='bold')

for row, label in enumerate(sorted(samples_by_label)):
    s    = samples_by_label[label]
    name = label_to_name[label]
    age  = str(s['bleed_age'])
    r_mm = float(s['bleed_radius_mm'])
    scat = s['signals_scattered']
    t_ns = np.arange(scat.shape[2]) * float(s['dt_s']) * 1e9

    ax = axes[row, 0]
    for rx in range(scat.shape[1]):
        ax.plot(t_ns, scat[0, rx], alpha=0.6, lw=0.8)
    title = name if age == 'none' else f'{name} ({age}, r={r_mm:.0f}mm)'
    ax.set(title=title, xlabel='Time (ns)', ylabel='Scattered Ez (V/m)')
    ax.grid(alpha=0.3)

    ax2 = axes[row, 1]
    das = s['das_image']
    im  = ax2.imshow(das, cmap='hot', origin='lower',
                     extent=[0, GRID_SIZE*DX_MM, 0, GRID_SIZE*DX_MM])
    ax2.set(title='DAS backprojection', xlabel='x (mm)', ylabel='y (mm)')
    if r_mm > 0:
        cx, cy = float(s['bleed_center_mm'][0]), float(s['bleed_center_mm'][1])
        ax2.plot(cx, cy, 'c+', markersize=15, mew=2, label='true bleed')
        ax2.legend(fontsize=8)
    plt.colorbar(im, ax=ax2)

plt.tight_layout()
os.makedirs('docs/assets', exist_ok=True)
plt.savefig('docs/assets/brain_dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/brain_dataset_samples.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Validate all 5 dataset quality rules                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import math
from datasets.brain.tissue_library import tissue_at_freq
from datasets.brain.phantom import PHANTOM_A, PHANTOM_B, sample_random_geometry

print('=== Dataset Quality Validation ===')

# Rule 1: Cole-Cole frequency-dependent
gm1, _ = tissue_at_freq('gray_matter', FREQ_GHZ * 1e9)
gm2, _ = tissue_at_freq('gray_matter', 2.4e9)
r1 = abs(gm1 - gm2) > 1.0
print(f'Rule 1 (Cole-Cole):   {gm1:.1f} @ {FREQ_GHZ}GHz vs {gm2:.1f} @ 2.4GHz  {"✓" if r1 else "✗"}')

# Rule 2: No bleeds in bone — check first 100 bleed samples
violations = 0
cx = cy = cz = GRID_SIZE // 2
for path, label in list(zip(train_manifest['sample_paths'], train_manifest['labels']))[:100]:
    if label == 0: continue
    s = np.load(path, allow_pickle=True)
    c = s['bleed_center_cells']
    d = math.sqrt(sum((int(c[i])-[cx,cy,cz][i])**2 for i in range(3)))
    skull_r = int(s['phantom_skull_inner_r'])
    if d > skull_r + 3: violations += 1
r2 = violations == 0
print(f'Rule 2 (Anatomy):     bone violations={violations}/100  {"✓" if r2 else "✗"}')

# Rule 3: Blood aging present
ages = set(a for a in train_manifest['bleed_ages'] if a != 'none')
r3 = len(ages) >= 2
print(f'Rule 3 (Blood aging): stages present={ages}  {"✓" if r3 else "✗"}')

# Rule 4a: Class balance
cc = train_manifest['class_counts']
tot = sum(cc.values())
fracs = {k: round(v/tot,2) for k,v in cc.items()}
r4a = all(0.15 <= f <= 0.40 for f in fracs.values())
print(f'Rule 4a (Balance):    fractions={fracs}  {"✓" if r4a else "✗"}')

# Rule 4b: Unique phantom per sample
skulls = []
for path in train_manifest['sample_paths'][:50]:
    s = np.load(path, allow_pickle=True)
    skulls.append(int(s['phantom_skull_inner_r']))
n_unique = len(set(skulls))
r4b = n_unique >= len(skulls) // 2  # at least half are distinct
print(f'Rule 4b (Unique phantom): {n_unique}/{len(skulls)} unique geometries  {"✓" if r4b else "✗"}')

all_ok = r1 and r2 and r3 and r4a and r4b
print()
print('✅ ALL 5 RULES PASS' if all_ok else '❌ Some rules failed')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Package for download                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil

OUT = pathlib.Path('/kaggle/working/waveforge_brain_outputs')
OUT.mkdir(exist_ok=True)
shutil.copy(manifest_path, OUT)
vis = pathlib.Path('docs/assets/brain_dataset_samples.png')
if vis.exists(): shutil.copy(vis, OUT)

train_files = sorted(TRAIN_DIR.glob('*.npz'))
test_files  = sorted(TEST_DIR.glob('*.npz'))
total_mb = sum(f.stat().st_size for f in train_files + test_files) / 1e6

print(f'Train samples: {len(train_files)}')
print(f'Test samples:  {len(test_files)}')
print(f'Total size:    {total_mb:.1f} MB')
print(f'Manifest:      {manifest_path}')
print('🏁 Dataset generation complete!')